In [31]:
## =========================================================
## Script: Step_Figures_Ferroptosis_Neurogenesis.R
## Starting from the annotated Seurat object, unify the color scheme and output a series of publication-quality figures (PDF + PNG, 300 dpi)
## =========================================================

library(Seurat)
library(dplyr)
library(ggplot2)

## ---------------- 0. Paths and objects ----------------
data_dir <- "~/mmFerroptosis/data/GSE233363"
fig_dir  <- file.path("~/mmFerroptosis/R_script", "figs")
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

# Object annotated with Celltype_Article
combined <- readRDS(
  file.path(data_dir, "Seurat_combined_with_Celltype_Article.rds")
)

DefaultAssay(combined) <- "RNA"

## ---------------- 0.1 Unify age information (Age, Age_collapsed) ----------------
meta_cols <- colnames(combined@meta.data)

if (!"Age" %in% meta_cols) {
  if ("timepoint" %in% meta_cols) {
    combined$Age <- combined$timepoint
  } else if ("GEMgroup" %in% meta_cols) {
    combined$Age <- combined$GEMgroup
  } else {
    stop("The object does not contain Age / timepoint / GEMgroup. Please confirm the age column name.")
  }
}

combined$Age_collapsed <- combined$Age
combined$Age_collapsed[combined$Age_collapsed %in% c("Old1", "Old2", "Old")] <- "Old"
combined$Age_collapsed[combined$Age_collapsed %in% c("Middle", "Middle-age", "MiddleAge")] <- "Middle"

combined$Age_collapsed <- factor(
  combined$Age_collapsed,
  levels = c("Young", "Middle", "Old")
)


In [32]:
## ---------------- 0.2 Color system ----------------

# Cell-type color palette (including the neurogenic lineage and others)
celltype_colors_full <- c(
  # --- Neurons (Blue/Teal spectrum) ---
  "qNSC"            = "#FDAE6B", # darkest blue
  "nIPC"            = "#3182BD", # medium blue
  "Neuroblast"      = "#08519C", # light blue
  "GC"              = "#9ECAE1", # pale blue (usually the most abundant cell type, so use a lighter color to avoid overpowering the plot)
  "Pyr"             = "#006D2C", # dark green (to distinguish it from granule cells)
  "Cajal-Retzius"   = "#74C476", # light green

  # --- Oligo Lineage (Orange spectrum) ---
  "OPC"             = "#E6550D", # dark orange
  "Oligodendrocyte" = "#FDAE6B", # light orange

  # --- Immune/Glial (Red/Pink spectrum) ---
  "Microglia"       = "#C51B8A", # rose red
  "PVM"             = "#F768A1", # pink
  "Astrocyte"       = "#D73027", # brick red

  # --- Vascular/Stromal (Grays & Purples) ---
  "Endothelial"     = "#542788", # dark purple
  "Pericyte"        = "#8073AC", # light purple
  "SMC"             = "#636363", # dark gray
  "VLMC"            = "#969696"  # light gray
)

# Age color palettes (multiple options)
age_colors_A <- c(
  "Young"  = "#1F78B4",
  "Middle" = "#A6CEE3",
  "Old"    = "#E31A1C"
)

age_colors_B <- c(
  "Young"  = "#4DBBD5",
  "Middle" = "#EFC000",
  "Old"    = "#D73027"
)

age_colors_C <- c(
  "Young"  = "#3288BD",
  "Middle" = "#ABDDDE",
  "Old"    = "#D53E4F"
)

age_palettes <- list(
  "A_simple" = age_colors_A,
  "B_neuro"  = age_colors_B,
  "C_nature" = age_colors_C
)

# Currently used age palette:
age_colors <- age_palettes[["B_neuro"]]

## ---------------- 0.3 Helper function: save figures ----------------

save_plot <- function(p, filename, width = 7, height = 6) {
  # 1. If the object is accidentally a patchwork object, remove the "patchwork" class
  if ("patchwork" %in% class(p)) {
    class(p) <- setdiff(class(p), "patchwork")
  }
  
  pdf_path <- file.path(fig_dir, paste0(filename, ".pdf"))
  png_path <- file.path(fig_dir, paste0(filename, ".png"))
  
  # 2. Open the graphics device and print in the most basic way to avoid ggsave handling lists oddly
  pdf(pdf_path, width = width, height = height)
  print(p)
  dev.off()
  
  png(png_path, width = width, height = height, units = "in", res = 300)
  print(p)
  dev.off()
  
  message("Saved: ", pdf_path, " and ", png_path)
}


In [33]:
## =========================================================
## Fig 1: UMAP of all cells (colored by Celltype_Article)
## =========================================================
combined <- RunUMAP(combined, dims = 1:30, seed.use = 123) 

## =========================================================
## Fig 1: UMAP of all cells (colored by Celltype_Article)
## =========================================================

Idents(combined) <- combined$Celltype_Article

p_umap_all <- DimPlot(
  combined,
  reduction = "umap",
  group.by  = "Celltype_Article",
  label     = FALSE
) + scale_color_manual(values = celltype_colors_full) +
  ggtitle("All cells – cell type annotation") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5),
    panel.grid = element_blank()
  )

save_plot(p_umap_all, "Fig1_AllCells_UMAP_Celltype", width = 7, height = 6)

09:24:19 UMAP embedding parameters a = 0.9922 b = 1.112

09:24:19 Read 37551 rows and found 30 numeric columns

09:24:19 Using Annoy for neighbor search, n_neighbors = 30

09:24:19 Building Annoy index with metric = cosine, n_trees = 50

0%   10   20   30   40   50   60   70   80   90   100%

[----|----|----|----|----|----|----|----|----|----|

*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
|

09:24:27 Writing NN index file to temp file /tmp/Rtmpn6Hz2e/file1124335e304fd

09:24:27 Searching Annoy index using 1 thread, search_k = 3000

09:24:44 Annoy recall = 100%

09:24:45 Commencing smooth kNN distance calibration using 1 thread
 with target n_neighbors = 30

09:24:48 Initializing from normalized Laplacian + noise (using irlba)

09:25:21 Commencing optimization for 200 epochs, with 1650878 positive edges

09:25:49 Optimization finished

Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion fail

In [34]:
## =========================================================
## Fig 1: UMAP of all cells (colored by Celltype_Article)
## - Title: total cell number
## - Legend: per-celltype counts
## =========================================================

## If UMAP has not been run yet (it already exists here, so you can keep it as is)
# combined <- RunUMAP(combined, dims = 1:30, seed.use = 123)

## ---------------- 1. Calculate cell counts ----------------

## Total cell count
total_n <- ncol(combined)

## Cell count for each Celltype
ct_n <- combined@meta.data %>%
  dplyr::count(Celltype_Article, name = "n_cells") %>%
  dplyr::mutate(
    legend_label = paste0(Celltype_Article, " (n=", n_cells, ")")
  )

## ---------------- 2. Construct the Celltype variable used for the legend ----------------

combined$Celltype_Article_legend <- as.character(combined$Celltype_Article)

for (i in seq_len(nrow(ct_n))) {
  ct <- as.character(ct_n$Celltype_Article[i])
  lb <- as.character(ct_n$legend_label[i])
  combined$Celltype_Article_legend[combined$Celltype_Article_legend == ct] <- lb
}

## Legend order: use celltype_colors_full as the reference to keep it stable
celltype_levels <- intersect(
  names(celltype_colors_full),
  ct_n$Celltype_Article
)

legend_levels <- ct_n$legend_label[
  match(celltype_levels, ct_n$Celltype_Article)
]

combined$Celltype_Article_legend <- factor(
  combined$Celltype_Article_legend,
  levels = legend_levels
)

## ---------------- 3. Color mapping (rename labels only, keep colors unchanged) ----------------

celltype_colors_legend <- celltype_colors_full[celltype_levels]
names(celltype_colors_legend) <- legend_levels

## ---------------- 4. Plotting ----------------

Idents(combined) <- combined$Celltype_Article_legend

p_umap_all <- DimPlot(
  combined,
  reduction = "umap",
  group.by  = "Celltype_Article_legend",
  label     = FALSE
) +
  scale_color_manual(values = celltype_colors_legend, name = "Cell type") +
  ggtitle(
    paste0(
      "All cells (n = ", format(total_n, big.mark = ","), 
      ") – cell type annotation"
    )
  ) +
  theme_bw() +
  theme(
    plot.title  = element_text(hjust = 0.5),
    panel.grid  = element_blank(),
    legend.text = element_text(size = 9)
  )

save_plot(
  p_umap_all,
  "Fig1_AllCells_UMAP_Celltype",
  width  = 7,
  height = 6
)


Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'All cells (n = 37,551) – cell type annotation' in 'mbcsToSbcs': dot substituted for <e2>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'All cells (n = 37,551) – cell type annotation' in 'mbcsToSbcs': dot substituted for <80>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'All cells (n = 37,551) – cell type annotation' in 'mbcsToSbcs': dot substituted for <93>”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'All cells (n = 37,551) – cell type annotation' in 'mbcsToSbcs': dot substituted for <e2>”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'All cells (n = 37,551) – cell type annotation' in 'mbcsToSbcs': dot substituted for <80>”
Warning messag

In [4]:
## =========================================================
## Fig 1: UMAP of all cells (colored by Celltype_Article)
## =========================================================

Idents(combined) <- combined$Celltype_Article

p_umap_all <- DimPlot(
  combined,
  reduction = "umap",
  group.by  = "Celltype_Article",
  label     = FALSE
) + scale_color_manual(values = celltype_colors_full) +
  ggtitle("All cells – cell type annotation") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5),
    panel.grid = element_blank()
  )

save_plot(p_umap_all, "Fig1_AllCells_UMAP_Celltype", width = 7, height = 6)

Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'All cells – cell type annotation' in 'mbcsToSbcs': dot substituted for <e2>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'All cells – cell type annotation' in 'mbcsToSbcs': dot substituted for <80>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'All cells – cell type annotation' in 'mbcsToSbcs': dot substituted for <93>”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'All cells – cell type annotation' in 'mbcsToSbcs': dot substituted for <e2>”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'All cells – cell type annotation' in 'mbcsToSbcs': dot substituted for <80>”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x

In [5]:
## =========================================================
## Fig 2: Neurogenic lineage subset (qNSC / nIPC / NB / GC / Astrocyte)
## =========================================================

neuro_ct <- c("qNSC", "nIPC", "Neuroblast", "GC", "Astrocyte")

neuro <- subset(
  combined,
  subset = Celltype_Article %in% neuro_ct
)

neuro$Celltype_Article <- factor(neuro$Celltype_Article, levels = neuro_ct)

p_umap_neuro <- DimPlot(
  neuro,
  reduction = "umap",
  group.by  = "Celltype_Article"
) + scale_color_manual(values = celltype_colors_full[neuro_ct]) +
  ggtitle("Neurogenic lineage + Astrocytes") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5),
    panel.grid = element_blank()
  )

save_plot(p_umap_neuro, "Fig2_Neurogenic_UMAP_Celltype", width = 7, height = 6)

# Facet by age
p_umap_neuro_age <- DimPlot(
  neuro,
  reduction = "umap",
  group.by  = "Celltype_Article",
  split.by  = "Age_collapsed"
) + scale_color_manual(values = celltype_colors_full[neuro_ct]) +
  ggtitle("Neurogenic lineage across ages") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5),
    panel.grid = element_blank()
  )

save_plot(p_umap_neuro_age, "Fig2_Neurogenic_UMAP_Celltype_byAge", width = 10, height = 6)

Saved: ~/mmFerroptosis/R_script/figs/Fig2_Neurogenic_UMAP_Celltype.pdf and ~/mmFerroptosis/R_script/figs/Fig2_Neurogenic_UMAP_Celltype.png

Saved: ~/mmFerroptosis/R_script/figs/Fig2_Neurogenic_UMAP_Celltype_byAge.pdf and ~/mmFerroptosis/R_script/figs/Fig2_Neurogenic_UMAP_Celltype_byAge.png



In [6]:
## =========================================================
## Fig 3: Age-related changes in cell composition (stacked bar)
## =========================================================

prop_df <- as.data.frame(table(neuro$Celltype_Article, neuro$Age_collapsed))
colnames(prop_df) <- c("Celltype", "Age", "Freq")

prop_df <- prop_df %>%
  group_by(Age) %>%
  mutate(Prop = Freq / sum(Freq))

p_bar_comp <- ggplot(prop_df, aes(x = Age, y = Prop, fill = Celltype)) +
  geom_bar(stat = "identity", color = "black", size = 0.2) +
  scale_fill_manual(values = celltype_colors_full[neuro_ct]) +
  scale_y_continuous(expand = c(0, 0)) +
  ylab("Proportion of neurogenic lineage cells") +
  xlab("Age") +
  ggtitle("Changes in neurogenic lineage composition across ages") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5),
    panel.grid = element_blank()
  )

save_plot(p_bar_comp, "Fig3_Neurogenic_Composition_byAge", width = 6, height = 5)

Warning message in geom_bar(stat = "identity", color = "black", size = 0.2):
“Ignoring unknown parameters: `size`”
Saved: ~/mmFerroptosis/R_script/figs/Fig3_Neurogenic_Composition_byAge.pdf and ~/mmFerroptosis/R_script/figs/Fig3_Neurogenic_Composition_byAge.png



In [7]:
## =========================================================
## Fig 4: Ferroptosis module score (calculate now if not already available)
## =========================================================

# Define the ferroptosis gene set (extend it further based on your curated table if needed)
ferro_genes_raw <- c(
  "Vdac2", "Vdac3",
  "Gpx4", "Hspb1",
  "Nfe2l2",
  "Nox1", "Nox4",
  "Trp53",
  "Slc7a11",
  "Terf1",
  "Ras",
  "Cars", "Eprs", "Hars",
  "Acsf2", "Emc2", "Rpl8", "Ireb2",
  "Cs", "Atp5g3",
  "Gclc",
  "Acsl4", "Lpcat3",
  "Gls2", "Got1",
  "Aco1",
  "Slc38a1",
  "G6pdx",
  "Ttc35",
  "Rb",
  "Chac1",
  "Ptges2",
  "Alox12",
  "Fth1", "Tfrc"
)

ferro_genes_raw <- unique(ferro_genes_raw)
genes_in_obj    <- rownames(neuro)
ferro_genes_use <- intersect(ferro_genes_raw, genes_in_obj)

# Calculate FerroScore only when needed
if (!any(grepl("^FerroScore", colnames(neuro@meta.data)))) {
  neuro <- AddModuleScore(
    object   = neuro,
    features = list(ferro_genes_use),
    name     = "FerroScore"
  )
}

ferro_col <- colnames(neuro@meta.data)[grepl("^FerroScore", colnames(neuro@meta.data))][1]

# Draw violin plots by cell type and age
p_vln_ferro <- VlnPlot(
  neuro,
  features = ferro_col,
  group.by = "Celltype_Article",
  split.by = "Age_collapsed",
  pt.size  = 0
) + scale_fill_manual(values = age_colors) +
  ggtitle("Ferroptosis module score by cell type and age") +
  theme(
    plot.title = element_text(hjust = 0.5)
  )

save_plot(p_vln_ferro, "Fig4_FerroScore_Violin_Celltype_byAge", width = 8, height = 5)

The default behaviour of split.by has changed.
Separate violin plots are now plotted side-by-side.
To restore the old behaviour of a single split violin,
set split.plot = TRUE.
      
This message will be shown once per session.



In [8]:
## =========================================================
## Fig 5: FerroScore summary (mean by Age × Celltype)
## =========================================================

ferro_df <- FetchData(
  neuro,
  vars = c(ferro_col, "Age_collapsed", "Celltype_Article")
)

colnames(ferro_df)[1] <- "FerroScore"

ferro_summary <- ferro_df %>%
  group_by(Celltype_Article, Age_collapsed) %>%
  summarise(
    mean_score = mean(FerroScore, na.rm = TRUE),
    sd_score   = sd(FerroScore, na.rm = TRUE),
    n_cells    = n(),
    .groups    = "drop"
  )

p_line_ferro <- ggplot(
  ferro_summary,
  aes(x = Age_collapsed, y = mean_score,
      group = Celltype_Article, color = Celltype_Article)
) +
  geom_line(size = 1) +
  geom_point(size = 2) +
  scale_color_manual(values = celltype_colors_full[neuro_ct]) +
  theme_bw() +
  xlab("Age") +
  ylab("Mean ferroptosis module score") +
  ggtitle("Ferroptosis score across ages in neurogenic lineage") +
  theme(
    plot.title = element_text(hjust = 0.5),
    panel.grid = element_blank()
  )

save_plot(p_line_ferro, "Fig5_FerroScore_Line_Celltype_byAge", width = 6, height = 5)


Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.”
Saved: ~/mmFerroptosis/R_script/figs/Fig5_FerroScore_Line_Celltype_byAge.pdf and ~/mmFerroptosis/R_script/figs/Fig5_FerroScore_Line_Celltype_byAge.png



In [9]:
## =========================================================
## Fig 6: Expression of key ferro genes (Slc7a11, Gpx4, Fth1, Tfrc)
## =========================================================

genes_example <- c("Slc7a11", "Gpx4", "Fth1", "Tfrc")
genes_example <- intersect(genes_example, rownames(neuro))

# Violin plot: age × cell type
p_vln_genes <- VlnPlot(
  neuro,
  features = genes_example,
  group.by = "Age_collapsed",
  split.by = "Celltype_Article",
  pt.size  = 0,
  combine  = TRUE
) + scale_fill_manual(values = celltype_colors_full[neuro_ct]) +
  ggtitle("Key ferroptosis genes across ages and cell types") +
  theme(
    plot.title = element_text(hjust = 0.5)
  )

save_plot(p_vln_genes, "Fig6_FerroGenes_Violin_Age_byCelltype", width = 10, height = 6)

## Optional: FeaturePlot on UMAP (single gene)
p_feat <- FeaturePlot(
  neuro,
  features = genes_example,
  split.by = "Age_collapsed",
  cols     = c("lightgrey", "red"),
  order    = TRUE
)

save_plot(p_feat, "Fig6_FerroGenes_FeaturePlot_UMAP", width = 9, height = 12)

In [10]:
## ===== 1. Convert Table S1 to mouse gene symbols (your provided version) =====
## Iron metabolism
ferro_iron_mouse <- c(
  "Fancd2", "Ncoa4", "Tfrc", "Phkg2", "Hsbp1",
  "Aco1", "Fth1", "Steap3", "Nfs1", "Ireb2",
  "Hmox1", "Mt1g"
)

## Lipid metabolism
ferro_lipid_mouse <- c(
  "Acsl4", "Akr1c1", "Akr1c2", "Akr1c3",
  "Alox15", "Alox5", "Alox12",
  "Cars", "Cbs", "Cisd1", "Cs", "Dpp4",
  "Hmgcr", "Gpx4", "Lpcat3", "Fdft1",
  "Acsl3", "Pebp1", "Zeb1", "Sqle", "Fads2",
  "Acsf2", "Ptgs2", "Acaca"
)

## (Anti)oxidant metabolism
ferro_antiox_mouse <- c(
  "Gclc", "Slc7a11", "Keap1", "Nqo1",
  "Abcc1", "Chac1", "Gss", "Gclm",
  "Nfe2l2", "Nox1"
)

## Energy metabolism
ferro_energy_mouse <- c(
  "Gls2", "Slc1a5", "Got1", "G6pd",
  "Pgd", "Atp5g3"
)

## Other
ferro_other_mouse <- c(
  "Cd44", "Hspb1", "Cryab", "Rpl8", "Sat1", "Tp53", "Emc2", "Aifm2"
)

## Merge into one ferro gene table
ferro_genes_mouse <- unique(c(
  ferro_iron_mouse,
  ferro_lipid_mouse,
  ferro_antiox_mouse,
  ferro_energy_mouse,
  ferro_other_mouse
))

## Keep only genes present in the object
genes_in_neuro   <- rownames(neuro)
ferro_genes_use  <- intersect(ferro_genes_mouse, genes_in_neuro)

message("Detected ferroptosis genes in neuro object: ",
        length(ferro_genes_use), " / ", length(ferro_genes_mouse))
print(sort(ferro_genes_use))

## Build the gene-to-functional-category mapping
gene_category <- c(
  setNames(rep("Iron metabolism",        length(ferro_iron_mouse)),   ferro_iron_mouse),
  setNames(rep("Lipid metabolism",       length(ferro_lipid_mouse)),  ferro_lipid_mouse),
  setNames(rep("(Anti)oxidant metabolism", length(ferro_antiox_mouse)), ferro_antiox_mouse),
  setNames(rep("Energy metabolism",      length(ferro_energy_mouse)), ferro_energy_mouse),
  setNames(rep("Other",                  length(ferro_other_mouse)),  ferro_other_mouse)
)

# Keep only the subset present in the object
gene_category_use <- gene_category[ferro_genes_use]


Detected ferroptosis genes in neuro object: 54 / 60



 [1] "Abcc1"   "Acaca"   "Aco1"    "Acsf2"   "Acsl3"   "Acsl4"   "Aifm2"  
 [8] "Alox12"  "Alox15"  "Alox5"   "Atp5g3"  "Cars"    "Cbs"     "Cd44"   
[15] "Chac1"   "Cisd1"   "Cryab"   "Cs"      "Dpp4"    "Emc2"    "Fads2"  
[22] "Fancd2"  "Fdft1"   "Fth1"    "Gclc"    "Gclm"    "Gls2"    "Got1"   
[29] "Gpx4"    "Gss"     "Hmgcr"   "Hmox1"   "Hsbp1"   "Hspb1"   "Ireb2"  
[36] "Keap1"   "Lpcat3"  "Ncoa4"   "Nfe2l2"  "Nfs1"    "Nox1"    "Nqo1"   
[43] "Pebp1"   "Pgd"     "Phkg2"   "Ptgs2"   "Rpl8"    "Sat1"    "Slc1a5" 
[50] "Slc7a11" "Sqle"    "Steap3"  "Tfrc"    "Zeb1"   


In [11]:
## ========================== Fig7. Age-pair DE heatmap ==========================
## For each cell type, perform the following:
##   1) Old vs Young
##   2) Middle vs Young
##   3) Old vs Middle
## Run differential analysis only on ferro_genes_use

library(dplyr)
library(tidyr)
library(ggplot2)

celltypes_to_test <- levels(neuro$Celltype_Article)

## Define the age pairs to compare (earlier, later)
age_pairs <- list(
  "Old_vs_Young"    = c("Young",  "Old"),
  "Middle_vs_Young" = c("Young",  "Middle"),
  "Old_vs_Middle"   = c("Middle", "Old")
)

de_list <- list()

for (ct in celltypes_to_test) {
  for (cmp_name in names(age_pairs)) {
    
    age_early <- age_pairs[[cmp_name]][1]
    age_late  <- age_pairs[[cmp_name]][2]
    
    message("Running DE (", cmp_name, ") for celltype: ", ct)
    
    sub <- subset(
      neuro,
      subset = Celltype_Article == ct & Age_collapsed %in% c(age_early, age_late)
    )
    
    ## Skip if there are too few cells
    if (ncol(sub) < 50) {
      warning("Too few cells for ", ct, " in comparison ", cmp_name, ", skip.")
      next
    }
    
    Idents(sub) <- sub$Age_collapsed
    
    ## ident.1 = later age, ident.2 = earlier age
    de_ct <- FindMarkers(
      sub,
      ident.1         = age_late,
      ident.2         = age_early,
      features        = ferro_genes_use,
      logfc.threshold = 0,      # do not pre-filter logFC
      min.pct         = 0.05    # expressed in at least 5% of cells
    )
    
    if (nrow(de_ct) == 0) {
      warning("No DE genes for ", ct, " in comparison ", cmp_name)
      next
    }
    
    de_ct$gene       <- rownames(de_ct)
    de_ct$Celltype   <- ct
    de_ct$contrast   <- cmp_name
    de_ct$later_age  <- age_late
    de_ct$earlier_age<- age_early
    
    de_list[[paste(ct, cmp_name, sep = "_")]] <- de_ct
  }
}

## Merge results from all celltype × comparison combinations
ferro_de_age <- bind_rows(de_list)

## Annotate significance and functional category
ferro_de_long <- ferro_de_age %>%
  mutate(
    sig      = ifelse(p_val_adj < 0.05 & abs(avg_log2FC) > 0.25, "yes", "no"),
    category = gene_category_use[gene]
  )

## Order genes by the maximum |log2FC| across all comparisons and cell types
gene_order_fc <- ferro_de_long %>%
  group_by(gene, category) %>%
  summarise(
    max_abs_fc = max(abs(avg_log2FC), na.rm = TRUE),
    .groups    = "drop"
  ) %>%
  arrange(category, -max_abs_fc) %>%
  pull(gene)

ferro_de_long$gene <- factor(ferro_de_long$gene, levels = rev(unique(gene_order_fc)))

## Fig7: log2FC heatmap (faceted by 3 comparisons) with significance markers -------------------------
p_fig7 <- ggplot(
  ferro_de_long,
  aes(x = Celltype, y = gene, fill = avg_log2FC)
) +
  geom_tile(color = NA) +
  geom_point(
    data = subset(ferro_de_long, sig == "yes"),
    aes(x = Celltype, y = gene),
    color  = "black",
    shape  = 21,
    size   = 1.2,
    stroke = 0.2
  ) +
  facet_wrap(~ contrast, nrow = 1) +
  scale_fill_gradient2(
    low      = "#2166AC",
    mid      = "white",
    high     = "#B2182B",
    midpoint = 0,
    name     = "log2FC (later vs earlier age)"
  ) +
  labs(
    x     = "Cell type",
    y     = "Ferroptosis-related genes",
    title = "Fig7. Age-associated changes in ferroptosis genes across neurogenic cell types"
  ) +
  theme_bw() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    axis.text.y = element_text(size = 6),
    panel.grid  = element_blank(),
    strip.background = element_rect(fill = "grey90", color = NA),
    strip.text       = element_text(face = "bold"),
    plot.title       = element_text(hjust = 0.5)
  )

save_plot(p_fig7, "Fig7_FerroGenes_log2FC_Heatmap_AllComparisons", width = 10, height = 8)


Running DE (Old_vs_Young) for celltype: qNSC

For a more efficient implementation of the Wilcoxon Rank Sum Test,
(default method for FindMarkers) please install the limma package
--------------------------------------------
install.packages('BiocManager')
BiocManager::install('limma')
--------------------------------------------
After installation of limma, Seurat will automatically use the more 
efficient implementation (no further action necessary).
This message will be shown once per session

Running DE (Middle_vs_Young) for celltype: qNSC

Running DE (Old_vs_Middle) for celltype: qNSC

Running DE (Old_vs_Young) for celltype: nIPC

Running DE (Middle_vs_Young) for celltype: nIPC

Running DE (Old_vs_Middle) for celltype: nIPC

Running DE (Old_vs_Young) for celltype: Neuroblast

Running DE (Middle_vs_Young) for celltype: Neuroblast

Running DE (Old_vs_Middle) for celltype: Neuroblast

Running DE (Old_vs_Young) for celltype: GC

Running DE (Middle_vs_Young) for celltype: GC

Running DE

In [12]:
## ========================= Fig8-1 / 8-2 / 8-3：hierarchical clustering heatmap (revised version) =========================
library(pheatmap)
library(dplyr)
library(tidyr)
library(grid)

# Use the same color gradient as above
hm_cols <- colorRampPalette(c("#2166AC", "white", "#B2182B"))(100)

# Assign IDs to the 3 comparisons
cmp_to_idx <- c(
  "Old_vs_Young"    = "1",
  "Middle_vs_Young" = "2",
  "Old_vs_Middle"   = "3"
)

dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

for (cmp_name in names(age_pairs)) {
  message("Drawing clustered heatmap for: ", cmp_name)
  
  sub_de <- ferro_de_long %>%
    dplyr::filter(contrast == cmp_name)
  
  if (nrow(sub_de) == 0) {
    warning("No DE results for comparison ", cmp_name, ", skip.")
    next
  }
  
  # log2FC matrix of gene × Celltype
  mat <- sub_de %>%
    dplyr::select(gene, Celltype, avg_log2FC) %>%
    tidyr::pivot_wider(
      id_cols     = gene,
      names_from  = Celltype,
      values_from = avg_log2FC
    ) %>%
    tibble::column_to_rownames("gene") %>%
    as.matrix()
  
  mat[is.na(mat)] <- 0   # treat NA as no change (0)
  
  # Row annotation: functional category
  ann_row <- data.frame(
    Category = gene_category_use[rownames(mat)]
  )
  rownames(ann_row) <- rownames(mat)
  
  age_early <- age_pairs[[cmp_name]][1]
  age_late  <- age_pairs[[cmp_name]][2]
  idx       <- cmp_to_idx[cmp_name]
  
  main_title <- paste0(
    "Fig8-", idx, ". ", gsub("_", " ", cmp_name),
    " (", age_late, " vs ", age_early, ")\n",
    "log2FC of ferroptosis-related genes"
  )
  
  base_name <- paste0("Fig8-", idx, "_FerroGenes_log2FC_Clustered_", cmp_name)
  pdf_file  <- file.path(fig_dir, paste0(base_name, ".pdf"))
  png_file  <- file.path(fig_dir, paste0(base_name, ".png"))
  
  ## Generate the pheatmap object first without drawing it directly to the device
  ph <- pheatmap::pheatmap(
    mat,
    color               = hm_cols,
    cluster_rows        = TRUE,
    cluster_cols        = TRUE,
    border_color        = NA,
    annotation_row      = ann_row,
    annotation_names_row= TRUE,
    show_rownames       = TRUE,
    show_colnames       = TRUE,
    fontsize_row        = 6,
    fontsize_col        = 9,
    main                = main_title,
    silent              = TRUE
  )
  
  ## ---- Save PDF ----
  pdf(pdf_file, width = 7, height = 7)
  grid::grid.newpage()
  grid::grid.draw(ph$gtable)
  dev.off()
  
  ## ---- Save PNG (300 dpi)----
  png(png_file, width = 7, height = 7, units = "in", res = 300)
  grid::grid.newpage()
  grid::grid.draw(ph$gtable)
  dev.off()
  
  message("Saved: ", pdf_file, " and ", png_file)
}


Drawing clustered heatmap for: Old_vs_Young

Saved: ~/mmFerroptosis/R_script/figs/Fig8-1_FerroGenes_log2FC_Clustered_Old_vs_Young.pdf and ~/mmFerroptosis/R_script/figs/Fig8-1_FerroGenes_log2FC_Clustered_Old_vs_Young.png

Drawing clustered heatmap for: Middle_vs_Young

Saved: ~/mmFerroptosis/R_script/figs/Fig8-2_FerroGenes_log2FC_Clustered_Middle_vs_Young.pdf and ~/mmFerroptosis/R_script/figs/Fig8-2_FerroGenes_log2FC_Clustered_Middle_vs_Young.png

Drawing clustered heatmap for: Old_vs_Middle

Saved: ~/mmFerroptosis/R_script/figs/Fig8-3_FerroGenes_log2FC_Clustered_Old_vs_Middle.pdf and ~/mmFerroptosis/R_script/figs/Fig8-3_FerroGenes_log2FC_Clustered_Old_vs_Middle.png



In [13]:
library(forcats)

## Calculate the maximum |log2FC| of each gene across all cell types
topN <- 15

ferro_rank <- ferro_de_long %>%
  group_by(gene, category) %>%
  summarise(
    max_abs_fc = max(abs(avg_log2FC), na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(-max_abs_fc) %>%
  slice_head(n = topN) %>%
  mutate(
    gene = fct_reorder(gene, max_abs_fc)
  )

p_fig9 <- ggplot(ferro_rank, aes(x = gene, y = max_abs_fc, fill = category)) +
  geom_col(color = "black", width = 0.7, size = 0.2) +
  coord_flip() +
  scale_fill_brewer(palette = "Set2", name = "Category") +
  labs(
    x = "Ferroptosis-related genes",
    y = "Max |log2FC| across neurogenic cell types (Old vs Young)",
    title = paste0("Fig9. Top ", topN, " age-sensitive ferroptosis genes in neurogenic lineage")
  ) +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5),
    panel.grid.major.y = element_blank()
  )

save_plot(p_fig9, "Fig9_FerroGenes_TopChanged_Barplot", width = 7, height = 6)


Warning message in geom_col(color = "black", width = 0.7, size = 0.2):
“Ignoring unknown parameters: `size`”
Saved: ~/mmFerroptosis/R_script/figs/Fig9_FerroGenes_TopChanged_Barplot.pdf and ~/mmFerroptosis/R_script/figs/Fig9_FerroGenes_TopChanged_Barplot.png



In [15]:
## ================== Basic settings ==================
library(dplyr)
library(ggplot2)
library(pheatmap)
library(igraph)
library(ggraph)

# Figure output directory
fig_dir  <- file.path("~/mmFerroptosis/R_script", "figs")
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

# Neurogenic order
neuro_order <- c("qNSC", "nIPC", "Neuroblast", "GC")

# Age palette (you have already defined age_palettes elsewhere)
# age_colors <- age_palettes[["B_neuro"]]

# General save function (ggplot / ggraph)
save_ggplot <- function(p, filename, width = 6, height = 5) {
  pdf_path <- file.path(fig_dir, paste0(filename, ".pdf"))
  png_path <- file.path(fig_dir, paste0(filename, ".png"))
  
  ggsave(pdf_path, plot = p, width = width, height = height, units = "in")
  ggsave(png_path, plot = p, width = width, height = height, units = "in", dpi = 300)
  
  message("Saved: ", pdf_path, " and ", png_path)

    
}


Attaching package: ‘igraph’


The following object is masked from ‘package:tidyr’:

    crossing


The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union




In [16]:
## ===================================================================
## Fig10. Neurogenic trajectory vs ferroptosis activity（line plot）
## ===================================================================
dfA <- neuro@meta.data %>%
  dplyr::filter(Celltype_Article %in% neuro_order) %>%
  group_by(Celltype_Article, Age_collapsed) %>%
  summarise(
    median_ferro = median(FerroScore1),
    se_ferro     = sd(FerroScore1) / sqrt(n()),
    .groups      = "drop"
  ) %>%
  mutate(Celltype_Article = factor(Celltype_Article, levels = neuro_order))

p10 <- ggplot(
  dfA,
  aes(
    x     = Celltype_Article,
    y     = median_ferro,
    color = Age_collapsed,
    group = Age_collapsed
  )
) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 2.5) +
  scale_color_manual(values = age_colors, name = "Age") +
  theme_bw() +
  labs(
    title = "Fig10. Neurogenic trajectory vs ferroptosis activity",
    x     = "Neurogenic lineage",
    y     = "Median ferroptosis score"
  ) +
  theme(
    plot.title  = element_text(hjust = 0.5),
    axis.title  = element_text(face = "bold"),
    panel.grid  = element_blank()
  )

save_ggplot(p10, "Fig10_NeurogenicTrajectory_FerroScore", width = 6.5, height = 4.5)

Saved: ~/mmFerroptosis/R_script/figs/Fig10_NeurogenicTrajectory_FerroScore.pdf and ~/mmFerroptosis/R_script/figs/Fig10_NeurogenicTrajectory_FerroScore.png



In [17]:
## ===================================================================
## Fig11. Pseudo-order neurogenic progression vs ferroptosis activity
## ===================================================================
dfB <- as.data.frame(neuro@meta.data)
dfB$UMAP_1 <- Embeddings(neuro, "umap")[, 1]

dfB2 <- dfB %>%
  dplyr::filter(Celltype_Article %in% neuro_order) %>%
  arrange(UMAP_1) %>%
  mutate(pseudo = row_number() / n())

p11 <- ggplot(
  dfB2,
  aes(
    x     = pseudo,
    y     = FerroScore1,
    color = Age_collapsed
  )
) +
  geom_point(alpha = 0.25, size = 0.6) +
  geom_smooth(method = "loess", se = TRUE, linewidth = 1.2) +
  scale_color_manual(values = age_colors, name = "Age") +
  theme_bw() +
  labs(
    title = "Fig11. Pseudo-neurogenic progression vs ferroptosis activity",
    x     = "Pseudo-neurogenic progression",
    y     = "Ferroptosis score"
  ) +
  theme(
    plot.title = element_text(hjust = 0.5),
    axis.title = element_text(face = "bold"),
    panel.grid = element_blank()
  )

save_ggplot(p11, "Fig11_PseudoOrder_FerroScore", width = 6.5, height = 4.5)

`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
Saved: ~/mmFerroptosis/R_script/figs/Fig11_PseudoOrder_FerroScore.pdf and ~/mmFerroptosis/R_script/figs/Fig11_PseudoOrder_FerroScore.png



In [18]:
## ===================================================================
## Fig12. Fraction of high-ferroptosis cells in neurogenic lineage
## ===================================================================
dfC <- neuro@meta.data %>%
  dplyr::filter(Celltype_Article %in% neuro_order) %>%
  group_by(Celltype_Article, Age_collapsed) %>%
  mutate(th = quantile(FerroScore1, 0.8)) %>%
  summarise(
    high_ratio = mean(FerroScore1 > th),
    .groups    = "drop"
  ) %>%
  mutate(Celltype_Article = factor(Celltype_Article, levels = neuro_order))

p12 <- ggplot(
  dfC,
  aes(
    x    = Celltype_Article,
    y    = high_ratio,
    fill = Age_collapsed
  )
) +
  geom_col(position = position_dodge(width = 0.7), width = 0.6) +
  scale_fill_manual(values = age_colors, name = "Age") +
  theme_bw() +
  labs(
    title = "Fig12. Fraction of high-ferroptosis cells in neurogenic lineage",
    x     = "Neurogenic lineage",
    y     = "Proportion of FerroScore-high cells"
  ) +
  theme(
    plot.title = element_text(hjust = 0.5),
    axis.title = element_text(face = "bold"),
    panel.grid = element_blank()
  )

save_ggplot(p12, "Fig12_HighFerroFraction_ByLineage", width = 6.5, height = 4.5)

Saved: ~/mmFerroptosis/R_script/figs/Fig12_HighFerroFraction_ByLineage.pdf and ~/mmFerroptosis/R_script/figs/Fig12_HighFerroFraction_ByLineage.png



In [27]:
## ===================================================================
## Fig13. Correlation heatmap: ferroptosis genes × neurogenesis markers
## ===================================================================
library(pheatmap)
library(grid)

## Neurogenesis marker genes (manually specify a set of classic markers)
neuro_markers <- c("Sox2","Hes1","Hopx","Ascl1","Dcx","Neurod1","Prox1","Calb2")

## Keep only the subset that actually exists in the current Seurat object
neuro_markers <- intersect(neuro_markers, rownames(neuro))

if (length(neuro_markers) == 0) {
  stop("None of the specified neurogenesis markers are present in rownames(neuro).")
}

# 1. Expression matrix
expr_df <- FetchData(neuro, vars = c(ferro_genes_use, neuro_markers))

# 2. Remove variables with sd == 0
sd_vec   <- apply(expr_df, 2, sd, na.rm = TRUE)
expr_df2 <- expr_df[, sd_vec > 0, drop = FALSE]

ferro_use2 <- intersect(ferro_genes_use, colnames(expr_df2))
neuro_use2 <- intersect(neuro_markers,    colnames(expr_df2))

if (length(ferro_use2) == 0 || length(neuro_use2) == 0) {
  stop("After filtering sd>0, no overlapping ferro or neurogenic markers remain.")
}

# 3. Spearman correlation
corr_mat <- cor(expr_df2, method = "spearman", use = "pairwise.complete.obs")

# 4. Submatrix: ferro × neuro
corr_sub <- corr_mat[ferro_use2, neuro_use2, drop = FALSE]

# 5. Remove rows/columns that are all NA
row_keep <- apply(corr_sub, 1, function(x) any(!is.na(x)))
col_keep <- apply(corr_sub, 2, function(x) any(!is.na(x)))
corr_sub <- corr_sub[row_keep, col_keep, drop = FALSE]

if (nrow(corr_sub) == 0 || ncol(corr_sub) == 0) {
  stop("corr_sub became empty after removing all-NA rows/cols.")
}

## 6. Generate the pheatmap object first, then draw it to the device using grid ------------
fig13_base <- "Fig13_Corr_Ferro_vs_NeuroMarkers"
fig13_pdf  <- file.path(fig_dir, paste0(fig13_base, ".pdf"))
fig13_png  <- file.path(fig_dir, paste0(fig13_base, ".png"))

# Color gradient
hm_cols <- colorRampPalette(c("#2166AC", "white", "#B2182B"))(100)

# Do not draw to the device yet; use silent = TRUE to obtain an object
ph13 <- pheatmap::pheatmap(
  corr_sub,
  color        = hm_cols,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  border_color = NA,
  main         = "Fig13. Correlation between ferroptosis genes and neurogenesis markers",
  silent       = TRUE
)

# ---- Save PDF ----
pdf(fig13_pdf, width = 6.5, height = 10)
grid::grid.newpage()
grid::grid.draw(ph13$gtable)
dev.off()

# ---- Save PNG (300 dpi)----
png(fig13_png, width = 6.5, height = 10, units = "in", res = 300)
grid::grid.newpage()
grid::grid.draw(ph13$gtable)
dev.off()

message("Saved: ", fig13_pdf, " and ", fig13_png)

pdf 
  2

pdf 
  2

Saved: ~/mmFerroptosis/R_script/figs/Fig13_Corr_Ferro_vs_NeuroMarkers.pdf and ~/mmFerroptosis/R_script/figs/Fig13_Corr_Ferro_vs_NeuroMarkers.png



In [28]:
## ===================================================================
## Fig14. Bipartite network: Ferroptosis genes ↔ Neurogenesis markers
## ===================================================================
edges <- as.data.frame(as.table(corr_sub)) %>%
  dplyr::rename(
    from   = Var1,  # ferro gene
    to     = Var2,  # neuro marker
    weight = Freq
  ) %>%
  dplyr::filter(!is.na(weight) & abs(weight) > 0.3)

if (nrow(edges) > 0) {
  g <- graph_from_data_frame(edges, directed = FALSE)
  
  # Mark the two sides of the bipartite graph
  all_vertices <- V(g)$name
  types <- all_vertices %in% rownames(corr_sub)   # TRUE: ferro, FALSE: neuro
  V(g)$type <- types
  
  # Colors and edge attributes
  V(g)$color <- ifelse(V(g)$type, "#E41A1C", "#377EB8")
  E(g)$width <- 1 + 2 * abs(E(g)$weight)
  E(g)$color <- ifelse(E(g)$weight > 0, "#B2182B", "#2166AC")
  
  layout_bi <- layout_as_bipartite(g, types = V(g)$type)
  
  fig14_base <- "Fig14_BipartiteNetwork_Ferro_vs_NeuroMarkers"
  fig14_pdf  <- file.path(fig_dir, paste0(fig14_base, ".pdf"))
  fig14_png  <- file.path(fig_dir, paste0(fig14_base, ".png"))
  
  # PDF
  pdf(fig14_pdf, width = 7, height = 5)
  plot(
    g,
    layout             = layout_bi,
    vertex.label       = V(g)$name,
    vertex.label.cex   = 0.6,
    vertex.label.color = "black",
    main               = "Fig14. Bipartite network: ferroptosis genes vs neurogenesis markers"
  )
  dev.off()
  
  # PNG
  png(fig14_png, width = 7, height = 5, units = "in", res = 300)
  plot(
    g,
    layout             = layout_bi,
    vertex.label       = V(g)$name,
    vertex.label.cex   = 0.6,
    vertex.label.color = "black",
    main               = "Fig14. Bipartite network: ferroptosis genes vs neurogenesis markers"
  )
  dev.off()
  
  message("Saved: ", fig14_pdf, " and ", fig14_png)
} else {
  warning("No edges with |correlation| > 0.3; Fig14 was not generated.")
}

Saved: ~/mmFerroptosis/R_script/figs/Fig14_BipartiteNetwork_Ferro_vs_NeuroMarkers.pdf and ~/mmFerroptosis/R_script/figs/Fig14_BipartiteNetwork_Ferro_vs_NeuroMarkers.png



In [21]:
## ===================================================================
## Fig15. Average ferroptosis burden along neurogenic lineage（bar plot）
## ===================================================================
dfF <- neuro@meta.data %>%
  dplyr::filter(Celltype_Article %in% neuro_order) %>%
  group_by(Celltype_Article, Age_collapsed) %>%
  summarise(
    ferro_mean = mean(FerroScore1),
    ferro_sd   = sd(FerroScore1),
    .groups    = "drop"
  ) %>%
  mutate(Celltype_Article = factor(Celltype_Article, levels = neuro_order))

p15 <- ggplot(
  dfF,
  aes(
    x    = Celltype_Article,
    y    = ferro_mean,
    fill = Age_collapsed
  )
) +
  geom_col(position = position_dodge(width = 0.7), width = 0.6) +
  geom_errorbar(
    aes(
      ymin = ferro_mean - ferro_sd,
      ymax = ferro_mean + ferro_sd
    ),
    position = position_dodge(width = 0.7),
    width    = 0.2
  ) +
  scale_fill_manual(values = age_colors, name = "Age") +
  theme_bw() +
  labs(
    title = "Fig15. Average ferroptosis burden along neurogenic lineage",
    x     = "Neurogenic lineage",
    y     = "Mean ferroptosis score ± SD"
  ) +
  theme(
    plot.title = element_text(hjust = 0.5),
    axis.title = element_text(face = "bold"),
    panel.grid = element_blank()
  )

save_ggplot(p15, "Fig15_MeanFerroScore_ByLineage", width = 6.5, height = 4.5)


Saved: ~/mmFerroptosis/R_script/figs/Fig15_MeanFerroScore_ByLineage.pdf and ~/mmFerroptosis/R_script/figs/Fig15_MeanFerroScore_ByLineage.png



In [30]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(pheatmap)
library(grid)

## =========================================================
## Fig7.1 & Fig7.2: Age comparisons (5 neuro_ct categories combined)
## =========================================================

## The 5 cell types to analyze (if you already have this vector, no need to redefine it)
neuro_cts <- c("qNSC", "nIPC", "Neuroblast", "GC", "Astrocyte")

## Consistently convert Age_collapsed into a factor for easier order control
neuro$Age_collapsed <- factor(neuro$Age_collapsed,
                              levels = c("Young", "Middle", "Old"))

##-------------------------------
## 1. Compute DE results for the three age comparisons
##-------------------------------
# Contrast names and the corresponding ident.1/ident.2
age_contrasts <- list(
  "Old_vs_Young"   = c("Old",   "Young"),
  "Old_vs_Middle"  = c("Old",   "Middle"),
  "Middle_vs_Young"= c("Middle","Young")
)

de_age_all_list <- list()

for (nm in names(age_contrasts)) {
  a1 <- age_contrasts[[nm]][1]
  a2 <- age_contrasts[[nm]][2]
  message("Running DE for contrast: ", nm, " (", a1, " vs ", a2, ")")
  
  sub <- subset(
    neuro,
    subset = Celltype_Article %in% neuro_cts & Age_collapsed %in% c(a1, a2)
  )
  
  if (ncol(sub) < 50) {
    warning("Too few cells for contrast ", nm, ", skip.")
    next
  }
  
  Idents(sub) <- sub$Age_collapsed
  
  de_tmp <- FindMarkers(
    sub,
    ident.1         = a1,
    ident.2         = a2,
    features        = ferro_genes_use,
    logfc.threshold = 0,      # do not pre-filter yet; inspect later
    min.pct         = 0.05
  )
  
  de_tmp$gene     <- rownames(de_tmp)
  de_tmp$contrast <- nm
  de_age_all_list[[nm]] <- de_tmp
}

ferro_de_age_all <- bind_rows(de_age_all_list)

# Add functional category
ferro_de_age_all$category <- gene_category_use[ferro_de_age_all$gene]

##-------------------------------
## 2. Fig7.1: Heatmap (log2FC, rows = genes, columns = contrast)
##-------------------------------

# Reshape into a gene × contrast matrix
de_mat_all <- ferro_de_age_all %>%
  select(gene, contrast, avg_log2FC) %>%
  tidyr::pivot_wider(
    id_cols     = gene,
    names_from  = contrast,
    values_from = avg_log2FC
  )

# Order by maximum |log2FC| and category
gene_order_fc_all <- ferro_de_age_all %>%
  group_by(gene) %>%
  summarise(
    category   = first(na.omit(category)),
    max_abs_fc = max(abs(avg_log2FC), na.rm = TRUE),
    .groups    = "drop"
  ) %>%
  arrange(category, desc(max_abs_fc)) %>%
  pull(gene)

# Convert to a matrix and arrange it in the order defined above
mat_all <- de_mat_all %>%
  tibble::column_to_rownames("gene") %>%
  as.matrix()

# Take the intersection to avoid NA row names
gene_order_fc_all <- intersect(gene_order_fc_all, rownames(mat_all))
mat_all <- mat_all[gene_order_fc_all, , drop = FALSE]

# Colors
hm_cols7 <- colorRampPalette(c("#2166AC", "white", "#B2182B"))(100)

# Generate the pheatmap object
ph7_1 <- pheatmap::pheatmap(
  mat_all,
  color        = hm_cols7,
  cluster_rows = TRUE,
  cluster_cols = FALSE,  # Only three columns are present, so display them directly in order
  border_color = NA,
  main         = "Fig7.1 Age-associated log2FC of ferroptosis genes\n(5 neurogenic cell types combined)",
  silent       = TRUE
)

# Save PDF + PNG
fig71_base <- "Fig7_1_FerroGenes_Log2FC_Heatmap_AgeComparisons"
fig71_pdf  <- file.path(fig_dir, paste0(fig71_base, ".pdf"))
fig71_png  <- file.path(fig_dir, paste0(fig71_base, ".png"))

pdf(fig71_pdf, width = 6.5, height = 7)
grid::grid.newpage()
grid::grid.draw(ph7_1$gtable)
dev.off()

png(fig71_png, width = 6.5, height = 7, units = "in", res = 300)
grid::grid.newpage()
grid::grid.draw(ph7_1$gtable)
dev.off()

message("Saved: ", fig71_pdf, " and ", fig71_png)

library(ggrepel)

## ---- Prepare volcano plot data ---------------------------------------------------
ferro_de_age_all2 <- ferro_de_age_all %>%
  mutate(
    neglog10p = -log10(p_val_adj + 1e-300),
    sig = ifelse(p_val_adj < 0.05 & abs(avg_log2FC) > 0.25, "Significant", "NS"),
    contrast = factor(
      contrast,
      levels = c("Old_vs_Young", "Old_vs_Middle", "Middle_vs_Young"),
      labels = c("Old vs Young", "Old vs Middle", "Middle vs Young")
    )
  )

# Control the y-axis height to avoid a few genes with extremely high -log10 p values
ymax_use <- quantile(ferro_de_age_all2$neglog10p, 0.99, na.rm = TRUE)
ferro_de_age_all2$neglog10p_capped <- pmin(ferro_de_age_all2$neglog10p, ymax_use)

# Significant genes to label
ferro_de_age_sig <- ferro_de_age_all2 %>%
  filter(sig == "Significant")


## ---- Draw volcano plots with gene labels --------------------------------------------
p7_2_labeled <- ggplot(
  ferro_de_age_all2,
  aes(x = avg_log2FC, y = neglog10p_capped)
) +
  geom_point(aes(color = sig), alpha = 0.8, size = 1.6) +
  scale_color_manual(
    values = c("NS" = "grey80", "Significant" = "#B2182B"),
    name   = ""
  ) +
  geom_vline(xintercept = c(-0.25, 0.25), 
             linetype = "dashed", color = "grey60", linewidth = 0.4) +
  geom_hline(yintercept = -log10(0.05), 
             linetype = "dashed", color = "grey60", linewidth = 0.4) +
  
  ## ---- Gene label annotation (key step)----
  geom_text_repel(
    data  = ferro_de_age_sig,
    aes(label = gene),
    size  = 3,
    max.overlaps = Inf,      # allow more labels
    box.padding = 0.35,
    point.padding = 0.2,
    segment.size = 0.2,
    min.segment.length = 0,
    segment.color = "grey50",
    color = "black"
  ) +
  
  facet_wrap(~ contrast, nrow = 1) +
  labs(
    title = "Fig7.2 Volcano plots of ferroptosis genes across age comparisons\n(5 neurogenic cell types combined)",
    x     = "log2FC",
    y     = "-log10(adj. P)"
  ) +
  theme_bw() +
  theme(
    plot.title  = element_text(hjust = 0.5),
    panel.grid  = element_blank(),
    strip.text  = element_text(face = "bold")
  )

save_plot(p7_2_labeled, "Fig7_2_FerroGenes_Volcano_AgeComparisons_Labeled",
          width = 9, height = 4)



Running DE for contrast: Old_vs_Young (Old vs Young)

Running DE for contrast: Old_vs_Middle (Old vs Middle)

Running DE for contrast: Middle_vs_Young (Middle vs Young)



pdf 
  2

pdf 
  2

Saved: ~/mmFerroptosis/R_script/figs/Fig7_1_FerroGenes_Log2FC_Heatmap_AgeComparisons.pdf and ~/mmFerroptosis/R_script/figs/Fig7_1_FerroGenes_Log2FC_Heatmap_AgeComparisons.png

Saved: ~/mmFerroptosis/R_script/figs/Fig7_2_FerroGenes_Volcano_AgeComparisons_Labeled.pdf and ~/mmFerroptosis/R_script/figs/Fig7_2_FerroGenes_Volcano_AgeComparisons_Labeled.png

